# Merged Detector (person, crosswalk, vehicles, speed bump)  trainv5
One YOLOv8 model, one class list, one forward pass per frame — replacing 3 separate detectors
(pedestrian+crosswalk, safety-distance vehicles, speed bump) with a single fine-tuned model.

**Merged classes:** `person`, `cross walk`, `car`, `truck`, `bus`, `Speed-Bump`, `Rumble Strip`

**Sources:**
- [`N_O_2`](https://universe.roboflow.com/nada-majd-e4uhb/n_o_2) (v23)  filtered down from its 70
  classes to just `person`, `cross walk`, `car`, `truck`, `bus`. Same filter-and-remap recipe as trainv3,
  extended to 5 classes instead of 2.
- [`speed-bump-detection-se0eh`](https://universe.roboflow.com/speed-bump-detection/speed-bump-detection-se0eh)
  — its 2 classes appended on top, images copied in separately (no overlap with N_O_2's images, so no
  merge conflicts — just two label sets sharing one `data.yaml`).

**NOT included:** traffic signs — you only have the pretrained sign model's weights (from the GitHub
repo), not its original training data, so there's nothing to merge it with. It stays a separate model in
both pipelines; you're only replacing the 3 detectors that had real underlying datasets.

Produces `merged_detector.pt` — same loading pattern as your other YOLO models.

*** UNVERIFIED, check before trusting: N_O_2's version (23, confirmed earlier) and the speed bump
dataset's version — check `project.versions()` for that one too before downloading. ***


## 1. Setup & imports

In [ ]:
!pip install ultralytics roboflow --quiet

import os
import shutil
import yaml
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Config

In [ ]:
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")

# Source 1: N_O_2 — pedestrian, crosswalk, vehicles
NO2_WORKSPACE = "nada-majd-e4uhb"
NO2_PROJECT = "n_o_2"
NO2_VERSION = 23  # confirmed earlier via project.versions()
NO2_RAW_DIR = "/kaggle/working/n_o_2_raw"

# Source 2: speed bump / rumble strip
BUMP_WORKSPACE = "speed-bump-detection"
BUMP_PROJECT = "speed-bump-detection-se0eh"
BUMP_VERSION = 13  # UNVERIFIED — confirm with project.versions() below, same as we did for N_O_2
BUMP_RAW_DIR = "/kaggle/working/speed_bump_raw"

DATASET_FORMAT = "yolov8"

# The 5 classes we keep from N_O_2's 70 (in this order = new ids 0-4)
NO2_CLASSES_TO_KEEP = ["person", "cross walk"]   # dropped car/truck/bus — not present in this N_O_2 version
MERGED_CLASSES = NO2_CLASSES_TO_KEEP + ["Speed-Bump", "Rumble Strip"]  # now 4 classes total

MERGED_DATA_DIR = "/kaggle/working/merged_dataset"

RUNS_DIR = "/kaggle/working/runs"
RUN_NAME = "merged_detector_yolo"

MODEL_SIZE = "yolov8n.pt"
IMG_SIZE = 640
EPOCHS = 80          # more classes / more data than any single previous run, give it more room
BATCH = 16
PATIENCE = 15


## 3. Download both source datasets
Confirm `BUMP_VERSION` with the cell below before downloading — same version-check habit as always.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

bump_project_check = rf.workspace(BUMP_WORKSPACE).project(BUMP_PROJECT)
print("speed-bump-detection-se0eh versions:")
for v in bump_project_check.versions():
    print(" ", v.version, v.name)


In [ ]:
# N_O_2
no2_project = rf.workspace(NO2_WORKSPACE).project(NO2_PROJECT)
no2_dataset = no2_project.version(NO2_VERSION).download(DATASET_FORMAT, location=NO2_RAW_DIR)
print("N_O_2 downloaded to:", no2_dataset.location)

# Speed bump
bump_project = rf.workspace(BUMP_WORKSPACE).project(BUMP_PROJECT)
bump_dataset = bump_project.version(BUMP_VERSION).download(DATASET_FORMAT, location=BUMP_RAW_DIR)
print("Speed bump dataset downloaded to:", bump_dataset.location)


## 3a. Discover both datasets' real class names
Confirm our target class strings match exactly what's in each `data.yaml` before filtering/remapping
anything — same check as every previous notebook.

In [ ]:
with open(os.path.join(no2_dataset.location, "data.yaml")) as f:
    no2_config = yaml.safe_load(f)
no2_names = no2_config["names"]
print(f"N_O_2: {len(no2_names)} classes")
print(no2_names)

no2_name_to_id = {v: int(k) for k, v in no2_names.items()} if isinstance(no2_names, dict) else {n: i for i, n in enumerate(no2_names)}
missing = [c for c in NO2_CLASSES_TO_KEEP if c not in no2_name_to_id]
if missing:
    print(f"\n*** WARNING: not found in N_O_2: {missing} — fix NO2_CLASSES_TO_KEEP above ***")
else:
    print(f"\nAll 5 target classes found. IDs: {[no2_name_to_id[c] for c in NO2_CLASSES_TO_KEEP]}")

print()
with open(os.path.join(bump_dataset.location, "data.yaml")) as f:
    bump_config = yaml.safe_load(f)
bump_names = bump_config["names"]
print(f"Speed bump dataset: {len(bump_names)} classes")
print(bump_names)


## 4. Build the merged dataset
Two steps: filter+remap N_O_2 down to 5 classes at ids 0-4, then copy the bump dataset's images/labels
in as-is but remapped to ids 5-6 — no image overlap between the two sources, so this is just two
label-remapping passes writing into the same folder tree.

In [ ]:
random.seed(42)

if os.path.exists(MERGED_DATA_DIR):
    shutil.rmtree(MERGED_DATA_DIR)
for split in ["train", "valid", "test"]:
    os.makedirs(os.path.join(MERGED_DATA_DIR, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(MERGED_DATA_DIR, split, "labels"), exist_ok=True)

# --- Step 1: N_O_2 -> ids 0-4 ---
no2_old_id_to_new_id = {no2_name_to_id[name]: new_id for new_id, name in enumerate(NO2_CLASSES_TO_KEEP)}

def merge_no2_split(split):
    src_img_dir = os.path.join(no2_dataset.location, split, "images")
    src_label_dir = os.path.join(no2_dataset.location, split, "labels")
    if not os.path.exists(src_img_dir):
        return 0, 0

    dst_img_dir = os.path.join(MERGED_DATA_DIR, split, "images")
    dst_label_dir = os.path.join(MERGED_DATA_DIR, split, "labels")

    img_files = [f for f in os.listdir(src_img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    kept_boxes = 0

    for img_file in img_files:
        label_file = os.path.splitext(img_file)[0] + ".txt"
        src_label_path = os.path.join(src_label_dir, label_file)

        filtered_lines = []
        if os.path.exists(src_label_path):
            with open(src_label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    old_id = int(parts[0])
                    if old_id in no2_old_id_to_new_id:
                        parts[0] = str(no2_old_id_to_new_id[old_id])
                        filtered_lines.append(" ".join(parts))

        shutil.copy(os.path.join(src_img_dir, img_file), os.path.join(dst_img_dir, f"no2_{img_file}"))
        with open(os.path.join(dst_label_dir, f"no2_{os.path.splitext(img_file)[0]}.txt"), "w") as f:
            f.write("\n".join(filtered_lines))

        kept_boxes += len(filtered_lines)

    return len(img_files), kept_boxes

print("Merging N_O_2 (person/cross walk/car/truck/bus):")
for split in ["train", "valid", "test"]:
    n_imgs, n_boxes = merge_no2_split(split)
    print(f"  {split}: {n_imgs} images, {n_boxes} boxes kept")


In [ ]:
# --- Step 2: speed bump dataset -> ids 5-6 ---
bump_old_id_to_new_id = {i: i + len(NO2_CLASSES_TO_KEEP) for i in range(len(bump_names))}

def merge_bump_split(split):
    src_img_dir = os.path.join(bump_dataset.location, split, "images")
    src_label_dir = os.path.join(bump_dataset.location, split, "labels")
    if not os.path.exists(src_img_dir):
        return 0, 0

    dst_img_dir = os.path.join(MERGED_DATA_DIR, split, "images")
    dst_label_dir = os.path.join(MERGED_DATA_DIR, split, "labels")

    img_files = [f for f in os.listdir(src_img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    kept_boxes = 0

    for img_file in img_files:
        label_file = os.path.splitext(img_file)[0] + ".txt"
        src_label_path = os.path.join(src_label_dir, label_file)

        remapped_lines = []
        if os.path.exists(src_label_path):
            with open(src_label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    old_id = int(parts[0])
                    parts[0] = str(bump_old_id_to_new_id[old_id])
                    remapped_lines.append(" ".join(parts))

        shutil.copy(os.path.join(src_img_dir, img_file), os.path.join(dst_img_dir, f"bump_{img_file}"))
        with open(os.path.join(dst_label_dir, f"bump_{os.path.splitext(img_file)[0]}.txt"), "w") as f:
            f.write("\n".join(remapped_lines))

        kept_boxes += len(remapped_lines)

    return len(img_files), kept_boxes

print("Merging speed bump dataset (Speed-Bump/Rumble Strip):")
for split in ["train", "valid", "test"]:
    n_imgs, n_boxes = merge_bump_split(split)
    print(f"  {split}: {n_imgs} images, {n_boxes} boxes kept")


## 4a. Write the merged data.yaml

In [ ]:
merged_data_yaml_path = os.path.join(MERGED_DATA_DIR, "data.yaml")
merged_config = {
    "train": os.path.join(MERGED_DATA_DIR, "train", "images"),
    "val": os.path.join(MERGED_DATA_DIR, "valid", "images"),
    "test": os.path.join(MERGED_DATA_DIR, "test", "images"),
    "nc": len(MERGED_CLASSES),
    "names": MERGED_CLASSES,
}
with open(merged_data_yaml_path, "w") as f:
    yaml.dump(merged_config, f)

print("Merged data.yaml:")
print(merged_config)


## 4b. Final counts before training

In [ ]:
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(MERGED_DATA_DIR, split, "images")
    count = len([f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    print(f"  {split}: {count} images")


## 5. Sanity-check a few merged, remapped labels before trusting anything
Mixing two sources means two chances to mess up the remapping — confirm boxes/classes line up
correctly on both a N_O_2-sourced image and a bump-sourced image.

In [ ]:
def preview_merged_labels(prefix, count=3):
    img_dir = os.path.join(MERGED_DATA_DIR, "train", "images")
    label_dir = os.path.join(MERGED_DATA_DIR, "train", "labels")

    candidates = [f for f in os.listdir(img_dir) if f.startswith(prefix)]
    candidates = [f for f in candidates if os.path.getsize(os.path.join(label_dir, os.path.splitext(f)[0] + ".txt")) > 0][:count]

    if not candidates:
        print(f"No labeled '{prefix}' images found to preview.")
        return

    fig, axes = plt.subplots(1, len(candidates), figsize=(5 * len(candidates), 5))
    if len(candidates) == 1:
        axes = [axes]

    for ax, img_file in zip(axes, candidates):
        img_path = os.path.join(img_dir, img_file)
        label_path = os.path.join(label_dir, os.path.splitext(img_file)[0] + ".txt")

        img = Image.open(img_path)
        w, h = img.size
        ax.imshow(img)

        with open(label_path) as f:
            for line in f:
                cls_id, xc, yc, bw, bh = map(float, line.strip().split())
                x1 = (xc - bw / 2) * w
                y1 = (yc - bh / 2) * h
                rect = patches.Rectangle((x1, y1), bw * w, bh * h, linewidth=2, edgecolor="lime", facecolor="none")
                ax.add_patch(rect)
                ax.text(x1, y1 - 5, MERGED_CLASSES[int(cls_id)], color="lime", fontsize=9, weight="bold")

        ax.set_title(img_file, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

preview_merged_labels("no2_", count=3)
preview_merged_labels("bump_", count=3)


## 6. Train (transfer learning on YOLOv8n, COCO-pretrained)

In [ ]:
model = YOLO(MODEL_SIZE)

results = model.train(
    data=merged_data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    project=RUNS_DIR,
    name=RUN_NAME,
    seed=42,
)


## 7. Plot training curves

In [ ]:
results_png = os.path.join(RUNS_DIR, RUN_NAME, "results.png")
if os.path.exists(results_png):
    img = Image.open(results_png)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
else:
    print(f"results.png not found at {results_png}")


## 8. Validate — check EVERY class, not just the average
A merged model can look fine overall while one class quietly underperforms.

In [ ]:
metrics = model.val()

print("Overall mAP50:", metrics.box.map50)
print("Overall mAP50-95:", metrics.box.map)
print("\nPer-class mAP50:")
for i, cls_name in enumerate(MERGED_CLASSES):
    try:
        print(f"  {cls_name}: {metrics.box.ap50[i]:.3f}")
    except (IndexError, KeyError):
        pass


## 9. Quick visual test on validation images

In [ ]:
best_weights = os.path.join(RUNS_DIR, RUN_NAME, "weights", "best.pt")
trained_model = YOLO(best_weights)

val_img_dir = os.path.join(MERGED_DATA_DIR, "valid", "images")
sample_files = sorted([f for f in os.listdir(val_img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])[:8]

for f in sample_files:
    img_path = os.path.join(val_img_dir, f)
    result = trained_model.predict(img_path, conf=0.4, verbose=False)[0]
    annotated = result.plot()[..., ::-1]
    plt.figure(figsize=(6, 6))
    plt.imshow(annotated)
    plt.title(f)
    plt.axis("off")
    plt.show()


## 10. Export weights
This is the one file that replaces 3 separate models (`ped_crosswalk_detector.pt`,
`speed_bump_detector.pt`, and the stock vehicle-detection call) in the merged pipeline version of
`fullwithsigns`.

In [ ]:
export_path = "/kaggle/working/merged_detector.pt"
shutil.copy(best_weights, export_path)
print(f"Saved to {export_path}")
print(f"Classes: {MERGED_CLASSES}")


## Next steps
1. Check section 8's per-class breakdown carefully — with 7 classes merged from 2 sources, it's easy
   for one class (probably `Rumble Strip`, likely the smallest) to quietly underperform while the
   average looks fine.
2. Once you're happy: I'll build `fullwithsigns_merged` — one loader for `merged_detector.pt`, one
   `detect_all()` function replacing 3 separate detector calls, feeding into the same
   `apply_overrides()` chain, alongside the still-separate road-type/weather classifiers and the
   still-separate traffic sign model.
3. Compare it against the original `fullwithsigns` on both **accuracy** (does merging hurt any single
   task?) and **latency** (time `classify_full()` end-to-end in both versions on the same test images)
   — that's the actual answer to whether merging was worth it.


In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

TEST_DIR = "/kaggle/input/datasets/khawlaelhamdi/testtest"

if os.path.exists(TEST_DIR):
    for f in os.listdir(TEST_DIR):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            image_path = os.path.join(TEST_DIR, f)

            result = trained_model.predict(image_path, conf=0.4, verbose=False)[0]
            annotated = result.plot()[..., ::-1]  # BGR -> RGB for matplotlib

            # Count detections per class for this image
            counts = {}
            for box in result.boxes:
                cls_name = trained_model.names[int(box.cls[0])]
                counts[cls_name] = counts.get(cls_name, 0) + 1

            plt.figure(figsize=(7, 7))
            plt.imshow(annotated)
            plt.title(f"{f}\n{counts if counts else 'no detections'}")
            plt.axis("off")
            plt.show()

            print(f"{f}: {counts if counts else 'no detections'}")
            print("-" * 50)
else:
    print(f"Error: {TEST_DIR} does not exist. Check the dataset path.")